# Labeled-case explorer

This notebook is the review surface for **stage 1**: attaching source records to each `candidate_record_id`.

It does not train a model. It answers:

1. Did the name/DOB linker attach the right rows?
2. For a labeled case, what evidence is actually on file at T0 and T1?
3. What does `review_warranted` vs `review_not_warranted` vs `insufficient_evidence` look like in the raw sources?

Matching rules and the wider pipeline are in `docs/DATA_FLOW.md`. If a rule here disagrees with that file, the code in `oos_review/linker.py` is the authority — update the doc after changing the code.


## Flow this notebook runs

```text
candidate_records.csv  -->  PersonIndex
                                |
Data_T0/*.csv  -----------------+-->  linked T0 tables
Data_T1/evidence_update_stream  -->  linked T1 table
                                |
                                v
                     dossier(candidate_id)
                                |
                                v
              side-by-side evidence for labeled cases
```

Every linked row carries `match_rule` so a miss is visible (unlinked / ambiguous) instead of silently dropped.


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "oos_review").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from oos_review.caseview import dossier, linkage_summary, rows_for_candidate
from oos_review.load import load_labels
from oos_review.pipeline import run_linkage

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

print("project root:", PROJECT_ROOT)


project root: /Users/surajk/Downloads/Identify_Out_of_State_Tag_Holders 2


## 1. Run the linker

`run_linkage` builds the candidate index, links every T0 source, then links the T1 stream (using T0 title `vehicle_ref` as a fallback). Artifacts are written to `outputs/linked/` for later stages.


In [3]:
bundle = run_linkage(save=True)
candidates = bundle["candidates"]
labels = load_labels()

source_names = [
    "address_history",
    "license_id_events",
    "vehicle_title_events",
    "work_location_signals",
    "external_context_signals",
    "evidence_update_stream",
]
print("candidates", len(candidates), "labeled", len(labels))


candidates 12000 labeled 300


## 2. Linkage coverage

A row is **linked** when it received exactly one `candidate_record_id`. Unlinked rows are expected: source files contain other people (family members, related records). Ambiguous rows are name collisions the linker refused to guess on.


In [4]:
coverage_rows = []
for name in source_names:
    summary = linkage_summary(bundle[name])
    summary.insert(0, "source", name)
    coverage_rows.append(summary)
coverage = pd.concat(coverage_rows, ignore_index=True)
coverage


,source,rows,linked,linked_share,unlinked,ambiguous,identity,dob_prefix,dob_initial,name_exact,name_prefix,vehicle_ref
0,address_history,48121,28671,0.595811,19019,431,0,0,0,26511,2160,0
1,license_id_events,48124,33026,0.686269,15090,8,26276,2092,3957,637,64,0
2,vehicle_title_events,48108,32309,0.671593,15346,453,0,0,0,26478,2232,3599
3,work_location_signals,24131,19020,0.788198,4823,288,0,0,0,17525,1495,0
4,external_context_signals,48116,28674,0.595935,18989,453,0,0,0,26526,2148,0
5,evidence_update_stream,24000,14859,0.619125,8909,232,0,0,0,13528,945,386


In [5]:
rule_tables = []
for name in source_names:
    counts = bundle[name]["match_rule"].value_counts().rename(name)
    rule_tables.append(counts)
pd.concat(rule_tables, axis=1).fillna(0).astype(int)


,address_history,license_id_events,vehicle_title_events,work_location_signals,external_context_signals,evidence_update_stream
match_rule,,,,,,
name_exact,26511,637,26478,17525,26526,13528
unlinked,19019,15090,15346,4823,18989,8909
name_prefix,2160,64,2232,1495,2148,945
ambiguous_unassigned,431,8,453,288,453,232
identity,0,26276,0,0,0,0
dob_initial,0,3957,0,0,0,0
dob_prefix,0,2092,0,0,0,0
vehicle_ref,0,0,3599,0,0,386


## 3. Development labels

300 cases have T0 and T1 labels. About one in five changes class after T1 evidence. Those changers are the best check that the T1 stream is actually being attached.


In [7]:
labeled = candidates.merge(labels, on="candidate_record_id", how="inner")
print(labeled["label_t0"].value_counts().to_string())
print()
print("T0 -> T1 transitions")
print(pd.crosstab(labeled["label_t0"], labeled["label_t1"]).to_string())
print()
print("class changes:", int((labeled["label_t0"] != labeled["label_t1"]).sum()))


label_t0
insufficient_evidence    105
review_not_warranted     105
review_warranted          90

T0 -> T1 transitions
label_t1               insufficient_evidence  review_not_warranted  review_warranted
label_t0                                                                            
insufficient_evidence                     71                    22                12
review_not_warranted                      20                    85                 0
review_warranted                          12                     1                77

class changes: 67


## 4. Case dossier

`show_case` prints the candidate snapshot, labels, and the linked evidence columns that matter for residency/registration review. `match_rule` is included so we can see whether a row was an exact identity match, a truncated given name, or a vehicle inheritance.


In [8]:
DISPLAY_COLS = {
    "address_history": [
        "state", "street_address", "effective_start_date", "effective_end_date",
        "source_type", "match_rule", "match_score",
    ],
    "license_id_events": [
        "credential_state", "event_type", "event_date", "credential_status",
        "date_of_birth", "match_rule", "match_score",
    ],
    "vehicle_title_events": [
        "vehicle_ref", "event_type", "event_state", "event_date",
        "match_rule", "match_score",
    ],
    "work_location_signals": [
        "work_state", "observed_date", "source_type", "match_rule", "match_score",
    ],
    "external_context_signals": [
        "signal_type", "signal_state", "effective_date", "evidence_quality",
        "source_description", "match_rule", "match_score",
    ],
    "evidence_update_stream": [
        "source_domain", "record_action", "state", "vehicle_ref",
        "effective_date", "observed_date", "source_description",
        "match_rule", "match_score",
    ],
}

LINKED_ONLY = {name: bundle[name] for name in source_names}


def show_case(candidate_record_id: str) -> dict:
    file = dossier(candidate_record_id, candidates, LINKED_ONLY, labels=labels)
    cand = file["candidate"]
    lab = file["label"] or {}
    print("=" * 72)
    print(candidate_record_id)
    print(
        f"  name={cand['first_name']} {cand['last_name']}  dob={cand['date_of_birth']}"
    )
    print(
        f"  observed_state={cand['observed_state']}  "
        f"observed={cand['candidate_observed_date']}  "
        f"status={cand['review_status']}"
    )
    print(
        f"  label_t0={lab.get('label_t0')}  label_t1={lab.get('label_t1')}  "
        f"counts={file['link_counts']}"
    )
    for name, frame in file["evidence"].items():
        cols = [c for c in DISPLAY_COLS[name] if c in frame.columns]
        print(f"\n--- {name} ({len(frame)} rows) ---")
        if frame.empty:
            print("  (no linked rows)")
            continue
        display_frame = frame[cols].sort_values(cols[0], kind="mergesort")
        print(display_frame.to_string(index=False))
    return file


## 5. One example of each class

These IDs come from `Development_Labels.csv`. Re-run after changing linker rules to see whether evidence appears or disappears.


In [9]:
EXAMPLES = {
    "review_warranted": "CAN-2B1096MPJS",
    "review_not_warranted": "CAN-1EJVSHCWNY",
    "insufficient_evidence": "CAN-QCF4GZN8TD",
    "t0_insufficient_to_t1_not_warranted": "CAN-O523BGV6IR",
    "t0_warranted_to_t1_insufficient": "CAN-Y1ONULRYLB",
}

dossiers = {label: show_case(cid) for label, cid in EXAMPLES.items()}


CAN-2B1096MPJS
  name=SYNGIV-Alectntl SYNFAM-Ytnlyzc  dob=SYNDOB-1988-11-24
  observed_state=PA  observed=2026-04-13  status=unreviewed
  label_t0=review_warranted  label_t1=review_warranted  counts={'address_history': 3, 'license_id_events': 3, 'vehicle_title_events': 3, 'work_location_signals': 2, 'external_context_signals': 2, 'evidence_update_stream': 1}

--- address_history (3 rows) ---
state                              street_address effective_start_date effective_end_date           source_type match_rule  match_score
   DE       SYNLOC-8502-ZLV-CZLO-312AA097CD805619           2025-06-26                NaN       account_profile name_exact         0.86
   PA     SYNLOC-370-MLJ-EPCCLNP-47D80544E9EA428A           2022-03-22         2023-12-01       customer_update name_exact         0.86
   PA SYNLOC-0776-RCLTYEP-LGPYFP-8E2F096AB7F6CBDF           2025-03-06         2025-05-16 correspondence_record name_exact         0.86

--- license_id_events (3 rows) ---
credential_state        e

## 6. Evidence volume by label

If `review_warranted` cases systematically have more DE-state rows, or `insufficient_evidence` cases have fewer linked rows, that becomes a feature in the next stage — not a conclusion yet.


In [10]:
def state_mix(frame: pd.DataFrame, state_col: str, labeled_ids: pd.Series) -> pd.DataFrame:
    subset = frame[frame["candidate_record_id"].isin(labeled_ids) & frame[state_col].notna()]
    return (
        subset.assign(is_de=subset[state_col].eq("DE"))
        .groupby("candidate_record_id")["is_de"]
        .mean()
    )


labeled_ids = labeled["candidate_record_id"]
features = labeled[["candidate_record_id", "observed_state", "label_t0", "label_t1"]].copy()
features["n_address"] = features["candidate_record_id"].map(
    bundle["address_history"].groupby("candidate_record_id").size()
).fillna(0)
features["n_license"] = features["candidate_record_id"].map(
    bundle["license_id_events"].groupby("candidate_record_id").size()
).fillna(0)
features["n_title"] = features["candidate_record_id"].map(
    bundle["vehicle_title_events"].groupby("candidate_record_id").size()
).fillna(0)
features["n_work"] = features["candidate_record_id"].map(
    bundle["work_location_signals"].groupby("candidate_record_id").size()
).fillna(0)
features["n_t1"] = features["candidate_record_id"].map(
    bundle["evidence_update_stream"].groupby("candidate_record_id").size()
).fillna(0)
features["addr_de_share"] = features["candidate_record_id"].map(
    state_mix(bundle["address_history"], "state", labeled_ids)
)
features["lic_de_share"] = features["candidate_record_id"].map(
    state_mix(bundle["license_id_events"], "credential_state", labeled_ids)
)

print("Mean linked-row counts by T0 label")
print(
    features.groupby("label_t0")[["n_address", "n_license", "n_title", "n_work", "n_t1"]]
    .mean()
    .round(2)
    .to_string()
)
print()
print("Mean share of DE rows among linked address / license events")
print(
    features.groupby("label_t0")[["addr_de_share", "lic_de_share"]]
    .mean()
    .round(3)
    .to_string()
)
print()
print("Labeled cases with zero linked rows in a source")
print(
    (features[["n_address", "n_license", "n_title", "n_work", "n_t1"]] == 0)
    .groupby(features["label_t0"])
    .mean()
    .round(3)
    .to_string()
)


Mean linked-row counts by T0 label
                       n_address  n_license  n_title  n_work  n_t1
label_t0                                                          
insufficient_evidence       2.43       2.77     2.77    1.66  1.30
review_not_warranted        2.61       2.83     2.75    1.69  1.26
review_warranted            2.46       2.74     2.72    1.50  1.23

Mean share of DE rows among linked address / license events
                       addr_de_share  lic_de_share
label_t0                                          
insufficient_evidence          0.534         0.483
review_not_warranted           0.491         0.470
review_warranted               0.510         0.498

Labeled cases with zero linked rows in a source
                       n_address  n_license  n_title  n_work   n_t1
label_t0                                                           
insufficient_evidence      0.019        0.0    0.000   0.029  0.133
review_not_warranted       0.010        0.0    0.010   0.029 

## 7. What to check by hand

When reading a dossier, ask:

- Is the observed state on the candidate the same as the latest address / license / title state?
- Are DE and non-DE facts both present, and do they conflict?
- Did T1 `record_correction` or `new_record` rows change that picture?
- Are too many rows unlinked (linker too strict) or clearly the wrong person (linker too loose)?

Next stage: freeze the linker rules unless the dossiers show systematic misses, then encode DE-tie / conflict / recency features on top of these linked tables.
